# Notebook 6: Improved Models — Targeting ROC-AUC ≥ 0.85

## Strategy Overview
| # | Strategy | Expected AUC Gain |
|---|---|---|
| 1 | Enhanced Feature Engineering (trends, rolling stats) | +5–7% |
| 2 | SMOTE Oversampling (fix class imbalance) | +3–5% |
| 3 | Tuned Random Forest (n=500 trees) | +3–5% |
| 4 | LightGBM (faster & stronger than XGBoost) | +3–5% |
| 5 | Bidirectional LSTM + Self-Attention | +3–5% |
| 6 | Soft Voting Ensemble (RF + LGB + LSTM) | +5–8% |

**Baseline**: Random Forest ROC-AUC = 0.6738  
**Target**: ROC-AUC ≥ 0.85


## Cell 1: Install Required Packages & Setup

In [ ]:
# Install required packages if not already installed
import subprocess, sys

packages = ['lightgbm', 'imbalanced-learn', 'shap', 'xgboost', 'scikit-learn', 'torch']
for pkg in packages:
    try:
        __import__(pkg.replace('-', '_').replace('imbalanced_learn', 'imblearn'))
        print(f'  [OK] {pkg} already installed')
    except ImportError:
        print(f'  [INSTALLING] {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f'  [DONE] {pkg} installed')

print('\nAll packages ready!')

  [OK] lightgbm already installed
  [OK] imbalanced-learn already installed
  [OK] shap already installed
  [OK] xgboost already installed
  [INSTALLING] scikit-learn...


In [ ]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ─── Paths ───────────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/ICU-Patient-Deterioration'
    print('Running in Google Colab.')
except ImportError:
    BASE_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
    print(f'Running Locally. Base dir: {BASE_DIR}')

DATA_DIR  = os.path.join(BASE_DIR, 'preprocessing_pipeline', 'output')
CSV_PATH  = os.path.join(DATA_DIR, 'processed_icu_dataset.csv')
MODEL_DIR = os.path.join(BASE_DIR, 'preprocessing_pipeline', 'output')

print(f'Data dir  : {DATA_DIR}')
print(f'CSV exists: {os.path.exists(CSV_PATH)}')

Running Locally. Base dir: c:\Users\LAPTOP\Downloads\database preproccesssing
Data dir  : c:\Users\LAPTOP\Downloads\database preproccesssing\preprocessing_pipeline\output
CSV exists: True


## Cell 2: Load Existing Sequence Data

In [ ]:
print('Loading existing .npy arrays...')
X_train = np.load(os.path.join(DATA_DIR, 'X_train.npy'))
X_val   = np.load(os.path.join(DATA_DIR, 'X_val.npy'))
X_test  = np.load(os.path.join(DATA_DIR, 'X_test.npy'))
y_train = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
y_val   = np.load(os.path.join(DATA_DIR, 'y_val.npy'))
y_test  = np.load(os.path.join(DATA_DIR, 'y_test.npy'))

print(f'X_train : {X_train.shape}   y_train: {y_train.shape}')
print(f'X_val   : {X_val.shape}   y_val  : {y_val.shape}')
print(f'X_test  : {X_test.shape}   y_test : {y_test.shape}')

# Class distribution
for name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    pos = y.sum(); total = len(y)
    print(f'{name}: {pos}/{total} positive ({pos/total*100:.2f}%) | Imbalance ratio: {(total-pos)/pos:.1f}:1')

Loading existing .npy arrays...
X_train : (64121, 24, 37)   y_train: (64121,)
X_val   : (15389, 24, 37)   y_val  : (15389,)
X_test  : (13667, 24, 37)   y_test : (13667,)
Train: 1872/64121 positive (2.92%) | Imbalance ratio: 33.3:1
Val: 447/15389 positive (2.90%) | Imbalance ratio: 33.4:1
Test: 560/13667 positive (4.10%) | Imbalance ratio: 23.4:1


## Cell 3: Strategy 1 — Enhanced Feature Engineering
We add **clinically meaningful time-series features** from the flat CSV:
- Rolling mean & std (6-hour window) for vitals
- Delta features (rate of change)
- Peak/min features
- Missing value flags (missingness is informative in ICU!)

In [ ]:
print('Loading flat CSV for feature engineering...')
print('(This may take 30-60 seconds for the large CSV)')

df = pd.read_csv(CSV_PATH)
print(f'Loaded CSV: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Columns: {list(df.columns[:15])} ...')

Loading flat CSV for feature engineering...
(This may take 30-60 seconds for the large CSV)


Loaded CSV: 138,953 rows × 47 columns
Columns: ['patientunitstayid', 'hour', 'heartrate', 'systemicsystolic', 'systemicdiastolic', 'systemicmean', 'respiration', 'temperature', 'Bilirubin', 'Creatinine', 'Glucose', 'Hemoglobin', 'Lactate', 'Platelets', 'Potassium'] ...


In [ ]:
# ─── Define vital columns to enhance ─────────────────────────────────────────
VITAL_COLS = ['heartrate', 'systemicsystolic', 'systemicdiastolic',
              'systemicmean', 'respiration', 'temperature']
LAB_COLS   = ['Glucose', 'Creatinine', 'Lactate', 'Hemoglobin',
              'Potassium', 'Sodium', 'WBC', 'Platelets', 'Bilirubin']

# Keep only cols that exist in CSV
VITAL_COLS = [c for c in VITAL_COLS if c in df.columns]
LAB_COLS   = [c for c in LAB_COLS if c in df.columns]

print(f'Vital cols found : {VITAL_COLS}')
print(f'Lab cols found   : {LAB_COLS}')

# Sort by patient and hour
df = df.sort_values(['patientunitstayid', 'hour']).reset_index(drop=True)

# ─── Add rolling features (per patient) ──────────────────────────────────────
print('\nAdding rolling features (per patient)...')
WINDOW = 6
new_features = []

for col in VITAL_COLS:
    if col in df.columns:
        grp = df.groupby('patientunitstayid')[col]
        df[f'{col}_roll6_mean'] = grp.transform(lambda x: x.rolling(WINDOW, min_periods=1).mean())
        df[f'{col}_roll6_std']  = grp.transform(lambda x: x.rolling(WINDOW, min_periods=1).std().fillna(0))
        df[f'{col}_delta']      = grp.transform(lambda x: x.diff().fillna(0))
        df[f'{col}_delta2']     = grp.transform(lambda x: x.diff().diff().fillna(0))  # acceleration
        new_features += [f'{col}_roll6_mean', f'{col}_roll6_std',
                         f'{col}_delta', f'{col}_delta2']

# Lab deltas
for col in ['Creatinine', 'Lactate', 'Glucose']:
    if col in df.columns:
        grp = df.groupby('patientunitstayid')[col]
        df[f'{col}_delta'] = grp.transform(lambda x: x.diff().fillna(0))
        new_features.append(f'{col}_delta')

# ─── Clinical composite features ──────────────────────────────────────────────
if 'heartrate' in df.columns and 'systemicsystolic' in df.columns:
    # Shock Index trend
    df['shock_index'] = df['heartrate'] / df['systemicsystolic'].replace(0, np.nan)
    df['shock_index'] = df['shock_index'].clip(0, 5).fillna(df['shock_index'].median())
    grp = df.groupby('patientunitstayid')['shock_index']
    df['shock_index_trend'] = grp.transform(lambda x: x.diff().fillna(0))
    new_features += ['shock_index', 'shock_index_trend']

if 'systemicsystolic' in df.columns and 'systemicdiastolic' in df.columns:
    # Pulse Pressure trend
    df['pulse_pressure_trend'] = df.groupby('patientunitstayid')[
        'pulse_pressure' if 'pulse_pressure' in df.columns else 'systemicsystolic'
    ].transform(lambda x: x.diff().fillna(0))
    new_features.append('pulse_pressure_trend')

print(f'Added {len(new_features)} new features')
print(f'New feature names: {new_features[:10]} ...')

Vital cols found : ['heartrate', 'systemicsystolic', 'systemicdiastolic', 'systemicmean', 'respiration', 'temperature']
Lab cols found   : ['Glucose', 'Creatinine', 'Lactate', 'Hemoglobin', 'Potassium', 'Sodium', 'WBC', 'Platelets', 'Bilirubin']

Adding rolling features (per patient)...
Added 30 new features
New feature names: ['heartrate_roll6_mean', 'heartrate_roll6_std', 'heartrate_delta', 'heartrate_delta2', 'systemicsystolic_roll6_mean', 'systemicsystolic_roll6_std', 'systemicsystolic_delta', 'systemicsystolic_delta2', 'systemicdiastolic_roll6_mean', 'systemicdiastolic_roll6_std'] ...


In [ ]:
# ─── Rebuild sequences with enhanced features ─────────────────────────────────
print('Rebuilding LSTM sequences with enhanced features...')

SEQUENCE_LENGTH = 24
RANDOM_SEED = 42

# Determine feature columns (same logic as pipeline step07)
exclude = {
    'patientunitstayid', 'hour',
    'unitdischargeoffset', 'unitdischargestatus', 'hospitaldischargestatus',
    'is_deteriorated', 'hours_until_discharge',
    'target_6h', 'target_12h', 'target_24h',
    'patienthealthsystemstayid', 'ethnicity', 'unitadmitsource',
    'unitvisitnumber', 'hospitaladmitoffset',
}
numeric_types = ['float64', 'float32', 'int64', 'int32', 'uint8', 'bool']
feature_cols = [
    c for c in df.columns
    if c not in exclude
    and str(df[c].dtype) in numeric_types
    and df[c].nunique() > 1  # remove constant columns
]

# Fill any NaN/Inf from new features
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())

print(f'Total features for enhanced model: {len(feature_cols)}')
print(f'Feature list: {feature_cols}')

# ─── Re-split patients with same seed ─────────────────────────────────────────
if 'target_24h' in df.columns:
    target_col = 'target_24h'
else:
    # Reconstruct from is_deteriorated + hours_until_discharge
    df['target_24h'] = (
        (df.get('is_deteriorated', 0) == 1) &
        (df.get('hours_until_discharge', 999) >= 0) &
        (df.get('hours_until_discharge', 999) <= 24)
    ).astype(int)
    target_col = 'target_24h'

patient_ids = df['patientunitstayid'].unique()
np.random.seed(RANDOM_SEED)
np.random.shuffle(patient_ids)
n = len(patient_ids)
train_ids = patient_ids[:int(n * 0.70)]
val_ids   = patient_ids[int(n * 0.70):int(n * 0.85)]
test_ids  = patient_ids[int(n * 0.85):]
print(f'Patients → Train:{len(train_ids)} | Val:{len(val_ids)} | Test:{len(test_ids)}')

def create_sequences(df, pid_list, feat_cols, tgt_col, seq_len=24):
    Xs, ys = [], []
    for pid in pid_list:
        p = df[df['patientunitstayid'] == pid].sort_values('hour')
        feats = p[feat_cols].values
        tgts  = p[tgt_col].values
        if len(feats) >= seq_len:
            for i in range(len(feats) - seq_len + 1):
                Xs.append(feats[i:i+seq_len])
                ys.append(tgts[i+seq_len-1])
        elif len(feats) > 0:
            pad = np.zeros((seq_len, feats.shape[1]))
            pad[-len(feats):] = feats
            Xs.append(pad)
            ys.append(tgts[-1])
    if not Xs:
        return np.empty((0, seq_len, len(feat_cols))), np.empty(0)
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.int32)

print('Creating enhanced sequences (may take 1-2 min)...')
Xe_train, ye_train = create_sequences(df, train_ids, feature_cols, target_col)
Xe_val,   ye_val   = create_sequences(df, val_ids,   feature_cols, target_col)
Xe_test,  ye_test  = create_sequences(df, test_ids,  feature_cols, target_col)

print(f'Enhanced X_train: {Xe_train.shape}, positives: {ye_train.sum()}/{len(ye_train)} ({ye_train.mean()*100:.1f}%)')
print(f'Enhanced X_val  : {Xe_val.shape},   positives: {ye_val.sum()}/{len(ye_val)} ({ye_val.mean()*100:.1f}%)')
print(f'Enhanced X_test : {Xe_test.shape},  positives: {ye_test.sum()}/{len(ye_test)} ({ye_test.mean()*100:.1f}%)')

Rebuilding LSTM sequences with enhanced features...
Total features for enhanced model: 66
Feature list: ['heartrate', 'systemicsystolic', 'systemicdiastolic', 'systemicmean', 'respiration', 'temperature', 'Bilirubin', 'Creatinine', 'Glucose', 'Hemoglobin', 'Lactate', 'Platelets', 'Potassium', 'Sodium', 'WBC', 'resp_fio2', 'resp_peep', 'age', 'diag_Diabetes', 'diag_Hypertension', 'diag_Heart_Disease', 'diag_Kidney_Disease', 'diag_Sepsis', 'diag_Pneumonia', 'diag_COPD', 'diag_Liver_Disease', 'med_Antibiotics', 'med_Vasopressors', 'med_Sedatives', 'med_Insulin', 'med_Anticoagulants', 'med_Steroids', 'bmi', 'pulse_pressure', 'map_calculated', 'shock_index', 'gender_Male', 'heartrate_roll6_mean', 'heartrate_roll6_std', 'heartrate_delta', 'heartrate_delta2', 'systemicsystolic_roll6_mean', 'systemicsystolic_roll6_std', 'systemicsystolic_delta', 'systemicsystolic_delta2', 'systemicdiastolic_roll6_mean', 'systemicdiastolic_roll6_std', 'systemicdiastolic_delta', 'systemicdiastolic_delta2', 'syst

## Cell 4: Strategy 2 — SMOTE Oversampling
Apply SMOTE **only on training data** (never val/test) to fix class imbalance.

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

N_train, seq_len, n_feats = Xe_train.shape
N_val   = Xe_val.shape[0]
N_test  = Xe_test.shape[0]

# Flatten for traditional ML
Xf_train = Xe_train.reshape(N_train, seq_len * n_feats)
Xf_val   = Xe_val.reshape(N_val,   seq_len * n_feats)
Xf_test  = Xe_test.reshape(N_test,  seq_len * n_feats)

print(f'Flattened train shape: {Xf_train.shape}')
print(f'Class distribution BEFORE SMOTE: {ye_train.sum()} pos / {len(ye_train)} total ({ye_train.mean()*100:.1f}%)')

# Apply SMOTE
# sampling_strategy=0.3 means: make minority class = 30% of majority
smote = SMOTE(sampling_strategy=0.3, random_state=42, k_neighbors=5)
try:
    Xf_train_sm, y_train_sm = smote.fit_resample(Xf_train, ye_train)
    print(f'Class distribution AFTER  SMOTE: {y_train_sm.sum()} pos / {len(y_train_sm)} total ({y_train_sm.mean()*100:.1f}%)')
except Exception as e:
    print(f'SMOTE failed ({e}), using original data with class weights instead.')
    Xf_train_sm, y_train_sm = Xf_train, ye_train

print(f'\nSMOTE-augmented train shape: {Xf_train_sm.shape}')

Flattened train shape: (64121, 1584)
Class distribution BEFORE SMOTE: 1872 pos / 64121 total (2.9%)
Class distribution AFTER  SMOTE: 18674 pos / 80923 total (23.1%)

SMOTE-augmented train shape: (80923, 1584)


## Cell 5: Strategy 3 — Tuned Random Forest (500 Trees)

In [ ]:
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

print('='*60)
print('STRATEGY 3: Tuned Random Forest (500 trees)')
print('='*60)

rf_tuned = RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    min_samples_leaf=5,
    min_samples_split=10,
    max_features='sqrt',
    class_weight='balanced_subsample',  # stronger than 'balanced'
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
print('Training Tuned RF on SMOTE-augmented data...')
rf_tuned.fit(Xf_train_sm, y_train_sm)
elapsed = time.time() - t0

rf_val_prob = rf_tuned.predict_proba(Xf_val)[:, 1]
rf_auc  = roc_auc_score(ye_val, rf_val_prob)

print(f'\n--- Tuned RF (500 trees) Validation Results ---')
print(f'  ROC-AUC : {rf_auc:.4f}  (baseline was 0.6738)')
print(f'  Training time: {elapsed:.1f}s')

# Store results
results = {'Tuned RF (500)': {'auc': rf_auc, 'probs_val': rf_val_prob}}

STRATEGY 3: Tuned Random Forest (500 trees)
Training Tuned RF on SMOTE-augmented data...

--- Tuned RF (500 trees) Validation Results ---
  ROC-AUC : 0.6860  (baseline was 0.6738)
  Training time: 56.8s


## Cell 6: Strategy 4 — LightGBM
LightGBM is faster and often more accurate than XGBoost for tabular data.

In [ ]:
import lightgbm as lgb

print('='*60)
print('STRATEGY 4: LightGBM')
print('='*60)

# Class weight ratio
neg_count = len(y_train_sm) - y_train_sm.sum()
pos_count = y_train_sm.sum()
scale_ratio = neg_count / max(pos_count, 1)
print(f'scale_pos_weight = {scale_ratio:.2f}')

lgb_model = lgb.LGBMClassifier(
    n_estimators=1000,
    num_leaves=127,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    is_unbalance=True,        # handles imbalance automatically
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

t0 = time.time()
print('Training LightGBM...')

# Use early stopping with eval set
callbacks = [lgb.early_stopping(stopping_rounds=50, verbose=False),
             lgb.log_evaluation(period=100)]

lgb_model.fit(
    Xf_train_sm, y_train_sm,
    eval_set=[(Xf_val, ye_val)],
    eval_metric='auc',
    callbacks=callbacks
)
elapsed = time.time() - t0

lgb_val_prob = lgb_model.predict_proba(Xf_val)[:, 1]
lgb_auc   = roc_auc_score(ye_val, lgb_val_prob)

print(f'\n--- LightGBM Validation Results ---')
print(f'  ROC-AUC : {lgb_auc:.4f}')
print(f'  Training time: {elapsed:.1f}s')

results['LightGBM'] = {'auc': lgb_auc, 'probs_val': lgb_val_prob}

STRATEGY 4: LightGBM
scale_pos_weight = 3.33
Training LightGBM...

--- LightGBM Validation Results ---
  ROC-AUC : 0.6543
  Training time: 13.0s


## Cell 7: Tuned XGBoost (with Best Params from Notebook 4)

In [ ]:
import xgboost as xgb

print('='*60)
print('Tuned XGBoost (improved params)')
print('='*60)

neg_orig = len(ye_train) - ye_train.sum()
pos_orig = ye_train.sum()
scale_pw  = neg_orig / max(pos_orig, 1)

xgb_tuned = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    scale_pos_weight=scale_pw,
    random_state=42,
    n_jobs=-1,
    eval_metric='auc',
    early_stopping_rounds=50,
    verbosity=0
)

t0 = time.time()
print(f'Training Tuned XGBoost (scale_pos_weight={scale_pw:.1f})...')
xgb_tuned.fit(
    Xf_train_sm, y_train_sm,
    eval_set=[(Xf_val, ye_val)],
    verbose=False
)
elapsed = time.time() - t0

xgb_val_prob = xgb_tuned.predict_proba(Xf_val)[:, 1]
xgb_auc   = roc_auc_score(ye_val, xgb_val_prob)

print(f'\n--- Tuned XGBoost Validation Results ---')
print(f'  ROC-AUC : {xgb_auc:.4f}')
print(f'  Training time: {elapsed:.1f}s')

results['Tuned XGBoost'] = {'auc': xgb_auc, 'probs_val': xgb_val_prob}

Tuned XGBoost (improved params)
Training Tuned XGBoost (scale_pos_weight=33.3)...

--- Tuned XGBoost Validation Results ---
  ROC-AUC : 0.6723
  Training time: 119.8s


## Cell 8: Strategy 5 — Bidirectional LSTM + Self-Attention

Architecture improvements over Notebook 5:
- **Bidirectional LSTM**: reads sequence forward AND backward
- **Self-Attention layer**: learns which timesteps matter most
- **LayerNorm**: stabilizes training
- **ReduceLROnPlateau**: drops learning rate when plateauing
- **Gradient clipping**: prevents exploding gradients
- **More features**: 37 → enhanced features

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ─── Model Architecture ───────────────────────────────────────────────────────
class SelfAttention(nn.Module):
    """Simple dot-product self-attention across timesteps."""
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Linear(hidden_size, 1)

    def forward(self, lstm_out):
        # lstm_out: (batch, seq_len, hidden)
        scores = self.attention(lstm_out)           # (batch, seq_len, 1)
        weights = torch.softmax(scores, dim=1)      # (batch, seq_len, 1)
        context = (weights * lstm_out).sum(dim=1)   # (batch, hidden)
        return context, weights


class BiLSTMAttention(nn.Module):
    """Bidirectional LSTM with Self-Attention for ICU deterioration prediction."""
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.bilstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.layer_norm = nn.LayerNorm(hidden_size * 2)  # *2 for bidirectional
        self.attention  = SelfAttention(hidden_size * 2)
        self.dropout    = nn.Dropout(dropout)
        self.fc1        = nn.Linear(hidden_size * 2, 64)
        self.relu       = nn.ReLU()
        self.fc2        = nn.Linear(64, 1)

    def forward(self, x):
        lstm_out, _ = self.bilstm(x)         # (batch, seq, hidden*2)
        lstm_out    = self.layer_norm(lstm_out)
        context, _  = self.attention(lstm_out)  # (batch, hidden*2)
        out = self.dropout(context)
        out = self.relu(self.fc1(out))
        out = self.dropout(out)
        out = self.fc2(out)
        return out.squeeze()


INPUT_SIZE  = Xe_train.shape[2]
HIDDEN_SIZE = 128
NUM_LAYERS  = 2
DROPOUT     = 0.3
LR          = 0.001
BATCH_SIZE  = 256
EPOCHS      = 50
PATIENCE    = 7

model = BiLSTMAttention(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal trainable parameters: {total_params:,}')

Using device: cpu
BiLSTMAttention(
  (bilstm): LSTM(66, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
  (attention): SelfAttention(
    (attention): Linear(in_features=256, out_features=1, bias=True)
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=256, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
)

Total trainable parameters: 613,250


In [ ]:
# ─── DataLoaders ──────────────────────────────────────────────────────────────
X_tr_t  = torch.tensor(Xe_train, dtype=torch.float32)
y_tr_t  = torch.tensor(ye_train, dtype=torch.float32)
X_val_t = torch.tensor(Xe_val, dtype=torch.float32)
y_val_t = torch.tensor(ye_val, dtype=torch.float32)
X_te_t  = torch.tensor(Xe_test, dtype=torch.float32)
y_te_t  = torch.tensor(ye_test, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t),  batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_te_t, y_te_t),  batch_size=BATCH_SIZE, shuffle=False)

# ─── Loss with class weight ───────────────────────────────────────────────────
pos_weight = float(len(ye_train) - int(ye_train.sum())) / float(max(int(ye_train.sum()), 1))
print(f'BCEWithLogitsLoss pos_weight = {pos_weight:.2f}')
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight, dtype=torch.float32).to(device))

optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

BCEWithLogitsLoss pos_weight = 33.25


TypeError: ReduceLROnPlateau.__init__() got an unexpected keyword argument 'verbose'

In [ ]:
import os
LSTM_MODEL_PATH = os.path.join(BASE_DIR, 'best_bilstm_model.pth')
if os.path.exists(LSTM_MODEL_PATH):
    print(f'Found existing BiLSTM model at {LSTM_MODEL_PATH}, skipping training.')
    model.load_state_dict(torch.load(LSTM_MODEL_PATH, map_location=device))
else:
    # ─── Training Loop ────────────────────────────────────────────────────────────
    print('='*60)
    print('STRATEGY 5: Bidirectional LSTM + Self-Attention Training')
    print('='*60)
    
    best_val_auc     = 0.0
    epochs_no_improve = 0
    history          = {'train_loss': [], 'val_loss': [], 'val_auc': []}
    
    for epoch in range(EPOCHS):
        # ── Train phase ──
        model.train()
        train_loss = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            preds = model(Xb)
            loss  = criterion(preds, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
            optimizer.step()
            train_loss += loss.item()
        avg_train_loss = train_loss / len(train_loader)
    
        # ── Validation phase ──
        model.eval()
        val_loss  = 0.0
        all_probs = []
        all_tgts  = []
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                preds  = model(Xb)
                loss   = criterion(preds, yb)
                val_loss += loss.item()
                probs  = torch.sigmoid(preds)
                all_probs.extend(probs.cpu().numpy())
                all_tgts.extend(yb.cpu().numpy())
    
        avg_val_loss = val_loss / len(val_loader)
        val_auc = roc_auc_score(all_tgts, all_probs)
    
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_auc'].append(val_auc)
    
        _prev_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_auc)
        if optimizer.param_groups[0]['lr'] < _prev_lr:
            print(f'  [LR] Reduced to {optimizer.param_groups[0]["lr"]:.6f}')
    
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | TrainLoss: {avg_train_loss:.4f} | ValLoss: {avg_val_loss:.4f} | ValAUC: {val_auc:.4f}')
    
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            epochs_no_improve = 0
            torch.save(model.state_dict(), LSTM_MODEL_PATH)
            print(f'  ✓ Best model saved (AUC={best_val_auc:.4f})')
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f'  Early stopping at epoch {epoch+1}!')
                break
    
    print(f'\nBest Validation AUC: {best_val_auc:.4f}')

In [ ]:
# ─── Final LSTM evaluation ─────────────────────────────────────────────────────
from sklearn.metrics import classification_report

model.load_state_dict(torch.load(LSTM_MODEL_PATH, map_location=device))
model.eval()

lstm_val_probs, lstm_val_tgts = [], []
with torch.no_grad():
    for Xb, yb in val_loader:
        Xb = Xb.to(device)
        preds = torch.sigmoid(model(Xb))
        lstm_val_probs.extend(preds.cpu().numpy())
        lstm_val_tgts.extend(yb.numpy())

lstm_val_probs = np.array(lstm_val_probs)
lstm_val_tgts  = np.array(lstm_val_tgts)

lstm_auc   = roc_auc_score(lstm_val_tgts, lstm_val_probs)

print(f'--- BiLSTM + Attention Validation Results ---')
print(f'  ROC-AUC : {lstm_auc:.4f}  (old LSTM was 0.6482)')

results['BiLSTM+Attention'] = {'auc': lstm_auc, 'probs_val': lstm_val_probs}

## Cell 9: Strategy 6 — Soft Voting Ensemble
Combine the 3 best models: **RF + LightGBM + BiLSTM**  
Weighted average (LightGBM gets highest weight as it's usually strongest)

In [ ]:
print('='*60)
print('STRATEGY 6: Soft Voting Ensemble')
print('='*60)

# Retrieve individual val probs
rf_probs   = results['Tuned RF (500)']['probs_val']
lgb_probs  = results['LightGBM']['probs_val']
xgb_probs  = results['Tuned XGBoost']['probs_val']
lstm_probs = results['BiLSTM+Attention']['probs_val']

# ── Find optimal weights using validation set ──────────────────────────────────
best_ensemble_auc = 0
best_weights = (0.25, 0.35, 0.20, 0.20)

# Grid search over weight combinations
for w1 in np.arange(0.1, 0.5, 0.1):       # RF weight
    for w2 in np.arange(0.2, 0.6, 0.1):   # LGB weight
        for w3 in np.arange(0.1, 0.4, 0.1):  # XGB weight
            w4 = 1.0 - w1 - w2 - w3
            if w4 < 0.05 or w4 > 0.5:
                continue
            ensemble = w1*rf_probs + w2*lgb_probs + w3*xgb_probs + w4*lstm_probs
            auc = roc_auc_score(ye_val, ensemble)
            if auc > best_ensemble_auc:
                best_ensemble_auc = auc
                best_weights = (w1, w2, w3, w4)

w1, w2, w3, w4 = best_weights
print(f'Optimal weights: RF={w1:.2f}, LGB={w2:.2f}, XGB={w3:.2f}, LSTM={w4:.2f}')

ensemble_val_probs = w1*rf_probs + w2*lgb_probs + w3*xgb_probs + w4*lstm_probs
ens_auc   = roc_auc_score(ye_val, ensemble_val_probs)

print(f'\n--- Ensemble Validation Results ---')
print(f'  ROC-AUC : {ens_auc:.4f}')

results['Ensemble (RF+LGB+XGB+LSTM)'] = {'auc': ens_auc, 'probs_val': ensemble_val_probs}

## Cell 10: ROC Curve Comparison — All Models

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
from sklearn.metrics import roc_curve

matplotlib.rcParams['figure.dpi'] = 120
plt.style.use('dark_background')

COLORS = [
    '#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4',
    '#FFEAA7', '#DDA0DD', '#FF8C94'
]

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('#1a1a2e')
for ax in axes:
    ax.set_facecolor('#16213e')

# ── ROC Curves ────────────────────────────────────────────────────────────────
ax1 = axes[0]
ax1.plot([0,1],[0,1],'--', color='gray', linewidth=1, label='Random (AUC=0.50)')
ax1.axvline(x=0, color='gray', linewidth=0.5)

# Baseline from Notebook 2 (stored result)
ax1.annotate('Baseline RF (NB2)\nAUC=0.6738', xy=(0.5, 0.60), fontsize=8, color='gray',
             ha='center', style='italic')

for i, (name, res) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(ye_val, res['probs_val'])
    lw = 3 if 'Ensemble' in name else 1.5
    ax1.plot(fpr, tpr, color=COLORS[i % len(COLORS)],
             linewidth=lw, label=f"{name} (AUC={res['auc']:.4f})")

ax1.set_xlabel('False Positive Rate', fontsize=11, color='white')
ax1.set_ylabel('True Positive Rate', fontsize=11, color='white')
ax1.set_title('ROC Curves — All Improved Models', fontsize=13, color='white', fontweight='bold')
ax1.legend(loc='lower right', fontsize=8, facecolor='#0f3460', edgecolor='white')
ax1.tick_params(colors='white')
ax1.spines['bottom'].set_color('gray')
ax1.spines['left'].set_color('gray')
ax1.grid(True, alpha=0.2)

# ── AUC Bar Chart ─────────────────────────────────────────────────────────────
ax2 = axes[1]

# Include baseline
all_names = ['Baseline RF\n(NB2)'] + [n.replace(' ', '\n') for n in results.keys()]
all_aucs  = [0.6738] + [r['auc'] for r in results.values()]
bar_colors = ['#666666'] + [COLORS[i % len(COLORS)] for i in range(len(results))]

bars = ax2.barh(all_names, all_aucs, color=bar_colors, edgecolor='white', linewidth=0.5, height=0.6)
ax2.axvline(x=0.6738, color='#666666', linestyle='--', linewidth=1.5, label='Baseline (0.6738)')
ax2.axvline(x=0.85,   color='#FFD700', linestyle='--', linewidth=1.5, alpha=0.7, label='Target (0.85)')
ax2.axvline(x=0.90,   color='#00FF7F', linestyle='--', linewidth=1.5, alpha=0.7, label='Goal (0.90)')

for bar, auc in zip(bars, all_aucs):
    ax2.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
             f'{auc:.4f}', va='center', ha='left', fontsize=9, color='white', fontweight='bold')

ax2.set_xlabel('ROC-AUC Score', fontsize=11, color='white')
ax2.set_title('ROC-AUC Comparison', fontsize=13, color='white', fontweight='bold')
ax2.set_xlim(0.5, 1.02)
ax2.tick_params(colors='white')
ax2.spines['bottom'].set_color('gray')
ax2.spines['left'].set_color('gray')
ax2.grid(True, axis='x', alpha=0.2)
ax2.legend(facecolor='#0f3460', edgecolor='white', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'preprocessing_pipeline', 'reports', 'roc_curves_improved.png'),
            dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('ROC curve saved to reports/roc_curves_improved.png')

## Cell 11: Final Comparison Table — All Models

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

print('='*70)
print('FINAL MODEL COMPARISON REPORT')
print('='*70)

# Build comparison DataFrame
baseline = {'Model': 'Baseline RF (Notebook 2)', 'Val ROC-AUC': 0.6738,
            'Improvement': '--'}
rows = [baseline]

for name, res in results.items():
    improvement = res['auc'] - 0.6738
    rows.append({
        'Model': name,
        'Val ROC-AUC': round(res['auc'], 4),
        4),
        'Improvement': f"+{improvement:.4f}" if improvement > 0 else f"{improvement:.4f}"
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

# Identify best model
best_name = max(results, key=lambda k: results[k]['auc'])
best_auc  = results[best_name]['auc']
best_probs = results[best_name]['probs_val']

print(f'\n>>> BEST MODEL: {best_name} with AUC = {best_auc:.4f}')
print(f'>>> Improvement over baseline: +{best_auc - 0.6738:.4f} ({(best_auc/0.6738 - 1)*100:.1f}% relative gain)')

# Save summary
summary_df.to_csv(
    os.path.join(BASE_DIR, 'preprocessing_pipeline', 'reports', 'model_comparison.csv'),
    index=False
)
print('\nSummary saved to reports/model_comparison.csv')

In [ ]:
# ─── Optimal Threshold Analysis for Best Model ────────────────────────────────
from sklearn.metrics import precision_recall_curve, f1_score

print('\n--- Optimal Threshold Analysis for Best Model ---')
precision, recall, thresholds = precision_recall_curve(ye_val, best_probs)
f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

print(f'Optimal threshold (max F1): {optimal_threshold:.3f}')
print(f'At this threshold:')

binary_preds = (best_probs >= optimal_threshold).astype(int)
cm = confusion_matrix(ye_val, binary_preds)
print(f'  Confusion Matrix:')
print(f'           Predicted 0  Predicted 1')
print(f'  Actual 0   {cm[0,0]:8d}   {cm[0,1]:8d}  (TN, FP)')
print(f'  Actual 1   {cm[1,0]:8d}   {cm[1,1]:8d}  (FN, TP)')
print(f'\n{classification_report(ye_val, binary_preds, target_names=["Stable", "Deteriorating"])}')

# ─── Final Evaluation on TEST SET ────────────────────────────────────────────
print('\n' + '='*60)
print('FINAL EVALUATION ON HELD-OUT TEST SET')
print('='*60)

# Get test probabilities for best model
if 'Ensemble' in best_name:
    # Recompute ensemble on test set
    rf_test  = rf_tuned.predict_proba(Xf_test)[:, 1]
    lgb_test = lgb_model.predict_proba(Xf_test)[:, 1]
    xgb_test = xgb_tuned.predict_proba(Xf_test)[:, 1]

    lstm_test_probs = []
    model.eval()
    with torch.no_grad():
        for Xb, _ in test_loader:
            Xb = Xb.to(device)
            p = torch.sigmoid(model(Xb))
            lstm_test_probs.extend(p.cpu().numpy())
    lstm_test = np.array(lstm_test_probs)

    best_test_probs = w1*rf_test + w2*lgb_test + w3*xgb_test + w4*lstm_test
elif 'LightGBM' in best_name:
    best_test_probs = lgb_model.predict_proba(Xf_test)[:, 1]
elif 'RF' in best_name:
    best_test_probs = rf_tuned.predict_proba(Xf_test)[:, 1]
elif 'XGBoost' in best_name:
    best_test_probs = xgb_tuned.predict_proba(Xf_test)[:, 1]
else:  # LSTM
    lstm_test_probs = []
    model.eval()
    with torch.no_grad():
        for Xb, _ in test_loader:
            Xb = Xb.to(device)
            p = torch.sigmoid(model(Xb))
            lstm_test_probs.extend(p.cpu().numpy())
    best_test_probs = np.array(lstm_test_probs)

test_auc   = roc_auc_score(ye_test, best_test_probs)
test_preds = (best_test_probs >= optimal_threshold).astype(int)

print(f'Best Model: {best_name}')
print(f'Test ROC-AUC : {test_auc:.4f}')
print(f'\nTest Set Classification Report (threshold={optimal_threshold:.3f}):')
print(classification_report(ye_test, test_preds, target_names=['Stable', 'Deteriorating']))

In [ ]:
# ─── Learning Curve Plot for BiLSTM ──────────────────────────────────────────
if history['val_auc']:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#1a1a2e')
    for ax in axes:
        ax.set_facecolor('#16213e')

    epochs_ran = range(1, len(history['val_auc']) + 1)

    axes[0].plot(epochs_ran, history['train_loss'], color='#FF6B6B', linewidth=2, label='Train Loss')
    axes[0].plot(epochs_ran, history['val_loss'],   color='#4ECDC4', linewidth=2, label='Val Loss')
    axes[0].set_title('BiLSTM Training Loss', color='white', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Epoch', color='white')
    axes[0].set_ylabel('Loss', color='white')
    axes[0].legend(facecolor='#0f3460', edgecolor='white')
    axes[0].tick_params(colors='white')
    axes[0].grid(True, alpha=0.2)

    axes[1].plot(epochs_ran, history['val_auc'], color='#FFEAA7', linewidth=2, label='Val AUC')
    axes[1].axhline(y=0.6482, color='gray', linestyle='--', linewidth=1, label='Old LSTM (0.6482)')
    axes[1].axhline(y=0.85,   color='#00FF7F', linestyle='--', linewidth=1, alpha=0.7, label='Target (0.85)')
    axes[1].set_title('BiLSTM Validation AUC', color='white', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Epoch', color='white')
    axes[1].set_ylabel('AUC', color='white')
    axes[1].legend(facecolor='#0f3460', edgecolor='white')
    axes[1].tick_params(colors='white')
    axes[1].grid(True, alpha=0.2)

    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, 'preprocessing_pipeline', 'reports', 'bilstm_training_curves.png'),
                dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
    plt.show()
    print('Training curves saved to reports/bilstm_training_curves.png')

In [ ]:
# ─── Feature Importance (LightGBM) ───────────────────────────────────────────
import matplotlib.pyplot as plt

print('\n--- Top 20 Feature Importances (LightGBM) ---')

# Reconstruct feature names for flattened input
n_timesteps = seq_len
n_raw_feats = len(feature_cols)

# For flat RF/LGB input (n_timesteps * n_raw_feats features)
flat_names = []
for t in range(n_timesteps):
    for f in feature_cols:
        flat_names.append(f'h{t}_{f}')

# Top 20 by LGB importance
try:
    importance = lgb_model.feature_importances_
    top_idx    = np.argsort(importance)[-20:][::-1]
    top_names  = [flat_names[i] if i < len(flat_names) else f'feat_{i}' for i in top_idx]
    top_vals   = importance[top_idx]

    # Get just the feature name (strip hour prefix)
    top_feat_names = ['_'.join(n.split('_')[1:]) for n in top_names]

    fig, ax = plt.subplots(figsize=(10, 7))
    fig.patch.set_facecolor('#1a1a2e')
    ax.set_facecolor('#16213e')

    bars = ax.barh(range(20), top_vals[::-1], color='#4ECDC4', edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(20))
    ax.set_yticklabels(top_feat_names[::-1], fontsize=9, color='white')
    ax.set_xlabel('LightGBM Feature Importance', color='white', fontsize=11)
    ax.set_title('Top 20 Most Important Features\n(LightGBM)', color='white', fontsize=13, fontweight='bold')
    ax.tick_params(colors='white')
    ax.grid(True, axis='x', alpha=0.2)
    ax.spines['bottom'].set_color('gray')
    ax.spines['left'].set_color('gray')

    plt.tight_layout()
    plt.savefig(os.path.join(BASE_DIR, 'preprocessing_pipeline', 'reports', 'lgbm_feature_importance.png'),
                dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
    plt.show()
    print('Feature importance saved to reports/lgbm_feature_importance.png')
except Exception as e:
    print(f'Could not plot feature importance: {e}')

In [ ]:
# ─── Final Summary ────────────────────────────────────────────────────────────
print('\n' + '█'*60)
print('  NOTEBOOK 6 COMPLETE — IMPROVEMENT SUMMARY')
print('█'*60)
print(f'\n  Baseline (RF Notebook 2) : ROC-AUC = 0.6738')
print(f'  Best improved model      : {best_name}')
print(f'  Best Val ROC-AUC         : {best_auc:.4f}')
print(f'  Best Test ROC-AUC        : {test_auc:.4f}')
print(f'  Total AUC gain           : +{best_auc - 0.6738:.4f}')
print(f'  Relative improvement     : {(best_auc/0.6738 - 1)*100:.1f}%')

print('\n  Saved files:')
print('    reports/roc_curves_improved.png')
print('    reports/bilstm_training_curves.png')
print('    reports/lgbm_feature_importance.png')
print('    reports/model_comparison.csv')
print('    best_bilstm_model.pth')

## Cell 12: Explainable AI (XAI) using SHAP
Generate SHAP explanations for the best-performing tree model (LightGBM) to understand which features drive the predictions.

In [ ]:
import shap
import matplotlib.pyplot as plt

print('='*60)
print('Explainable AI (XAI) - SHAP Analysis for LightGBM')
print('='*60)

try:
    # Initialize JS visualization for notebook
    shap.initjs()
    
    # Reconstruct feature names for flattened input
    n_timesteps = SEQUENCE_LENGTH
    flat_names = []
    for t in range(n_timesteps):
        for f in feature_cols:
            flat_names.append(f'h{t}_{f}')
            
    # Using a sample of test data for SHAP to save time
    Xf_test_sample = Xf_test[:500]
    
    # Explain the LightGBM model
    print('Calculating SHAP values (this may take a minute)...')
    explainer = shap.TreeExplainer(lgb_model)
    shap_values = explainer.shap_values(Xf_test_sample)
    
    # For LightGBM binary classification, shap_values is a list of length 2 (for class 0 and class 1)
    # We want the explanations for the positive class (class 1)
    if isinstance(shap_values, list):
        shap_values_pos = shap_values[1]
    else:
        shap_values_pos = shap_values
        
    # 1. Summary Plot (Beeswarm)
    plt.figure(figsize=(10, 6))
    # Temporarily switch to default style for SHAP as it doesn't play well with dark backgrounds
    plt.style.use('default') 
    shap.summary_plot(shap_values_pos, Xf_test_sample, feature_names=flat_names, show=False, max_display=15)
    plt.title('SHAP Summary Plot (Top 15 Features)', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    shap_path = os.path.join(BASE_DIR, 'preprocessing_pipeline', 'reports', 'shap_summary.png')
    plt.savefig(shap_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'SHAP summary plot saved to {shap_path}')
    
    # Revert to dark background for future plots
    plt.style.use('dark_background')
    
except Exception as e:
    print(f'Error generating SHAP explanations: {e}')